In [ ]:
# 实现一个简单的ListADT
class MyListADT:
    def __init__(self):
        # code
    def insert(self,value): # 末尾插入
        pass
    def delete(self,index): # 删除指定index的元素
        pass
    def get(self,index): # 返回指定index处的元素
        pass
    def size(self): # 返回当前长度
        pass

# ex1
class MyListADT:
    def __init__(self):
        self._data = [] # 空list

    def size(self):
        return len(self._data)



In [1]:
# Quick test: run notebook code in order
import random

class Stack:
    def __init__(self):
        self._items = []
    def push(self, item): self._items.append(item)
    def pop(self): return self._items.pop() if self._items else None
    def isEmpty(self): return len(self._items) == 0

class Cell:
    def __init__(self, row_index, col_index):
        self.row_index, self.col_index = row_index, col_index
        self.N = self.S = self.E = self.W = True
    def coordinate(self): return (self.row_index, self.col_index)
    def breakWall(self, direction):
        if direction == "N": self.N = False
        elif direction == "S": self.S = False
        elif direction == "E": self.E = False
        elif direction == "W": self.W = False

def getNeighbors(maze, cell, width, height):
    r, c = cell.coordinate()
    neighbors = []
    if r > 0: neighbors.append((maze[r-1][c], "N"))
    if r < height-1: neighbors.append((maze[r+1][c], "S"))
    if c > 0: neighbors.append((maze[r][c-1], "W"))
    if c < width-1: neighbors.append((maze[r][c+1], "E"))
    return neighbors

def breakWallBetweenTwoCells(u, v):
    ur, uc = u.coordinate()
    vr, vc = v.coordinate()
    if vr == ur-1 and vc == uc: u.breakWall("N"); v.breakWall("S")
    elif vr == ur+1 and vc == uc: u.breakWall("S"); v.breakWall("N")
    elif vr == ur and vc == uc-1: u.breakWall("W"); v.breakWall("E")
    elif vr == ur and vc == uc+1: u.breakWall("E"); v.breakWall("W")

def draw_maze(maze, width, height):
    for r in range(height):
        line_top = "+"
        for c in range(width):
            line_top += "--" if maze[r][c].N else "  "
            line_top += "+"
        print(line_top)
        line_mid = ""
        for c in range(width):
            line_mid += "|" if maze[r][c].W else " "
            line_mid += "  "
        line_mid += "|" if maze[r][width - 1].E else " "
        print(line_mid)
    line_bot = "+"
    for c in range(width):
        line_bot += "--" if maze[height - 1][c].S else "  "
        line_bot += "+"
    print(line_bot)

def maze_generation(width, height):
    maze = [[Cell(r, c) for c in range(width)] for r in range(height)]
    visited = [[False] * width for _ in range(height)]
    stack = Stack()
    stack.push(maze[0][0])
    visited[0][0] = True
    while not stack.isEmpty():
        current = stack.pop()
        unvisited = [(nb, d) for nb, d in getNeighbors(maze, current, width, height) if not visited[nb.row_index][nb.col_index]]
        if unvisited:
            stack.push(current)
            next_cell, direction = random.choice(unvisited)
            breakWallBetweenTwoCells(current, next_cell)
            visited[next_cell.row_index][next_cell.col_index] = True
            stack.push(next_cell)
    return maze

class Node:
    def __init__(self, coord): self.coord = coord; self.N = self.S = self.E = self.W = None
    def get_next_and_direction(self):
        if self.N: return (self.N, "N")
        if self.S: return (self.S, "S")
        if self.E: return (self.E, "E")
        if self.W: return (self.W, "W")
        return (None, None)

class LinkedList:
    def __init__(self): self.head = None
    def get_tail(self):
        if self.head is None: return None
        node = self.head
        while True:
            n, _ = node.get_next_and_direction()
            if n is None: return node
            node = n
    def get_parent_of_tail(self):
        if self.head is None: return None, None
        next_n, _ = self.head.get_next_and_direction()
        if next_n is None: return None, None
        prev, node = self.head, next_n
        while True:
            next_n, _ = node.get_next_and_direction()
            if next_n is None:
                if prev.N is node: return prev, "N"
                if prev.S is node: return prev, "S"
                if prev.E is node: return prev, "E"
                if prev.W is node: return prev, "W"
                return prev, None
            prev, node = node, next_n
    def __str__(self):
        if self.head is None: return ""
        parts = []; node = self.head
        r, c = node.coord; parts.append(f"({r}, {c})")
        while True:
            next_n, direction = node.get_next_and_direction()
            if next_n is None: break
            parts.append(f"+-{direction}-+"); r, c = next_n.coord; parts.append(f"({r}, {c})"); node = next_n
        return "".join(parts)
    def add(self, newNode):
        if self.head is None: self.head = newNode; return
        tail = self.get_tail(); tr, tc = tail.coord; nr, nc = newNode.coord
        dr, dc = nr - tr, nc - tc
        if dr == -1 and dc == 0: tail.N = newNode
        elif dr == 1 and dc == 0: tail.S = newNode
        elif dr == 0 and dc == -1: tail.W = newNode
        elif dr == 0 and dc == 1: tail.E = newNode
    def removeLastNode(self):
        if self.head is None: return
        if self.head.get_next_and_direction()[0] is None: self.head = None; return
        parent, direction = self.get_parent_of_tail()
        if direction == "N": parent.N = None
        elif direction == "S": parent.S = None
        elif direction == "E": parent.E = None
        elif direction == "W": parent.W = None

def maze_resolution(maze):
    height, width = len(maze), len(maze[0])
    end = (height - 1, width - 1)
    path = LinkedList(); path.add(Node((0, 0)))
    visited = {(0, 0)}
    while True:
        tail = path.get_tail()
        if tail is None: return ""
        if tail.coord == end: return path.__str__()
        r, c = tail.coord
        candidates = []
        if r > 0 and not maze[r][c].N and (r-1, c) not in visited: candidates.append((r-1, c))
        if r < height-1 and not maze[r][c].S and (r+1, c) not in visited: candidates.append((r+1, c))
        if c > 0 and not maze[r][c].W and (r, c-1) not in visited: candidates.append((r, c-1))
        if c < width-1 and not maze[r][c].E and (r, c+1) not in visited: candidates.append((r, c+1))
        if not candidates:
            visited.discard(tail.coord)
            path.removeLastNode()
            continue
        next_coord = candidates[0]
        visited.add(next_coord)
        path.add(Node(next_coord))

if __name__ == "__main__":
    print("--- draw_maze 2x2 ---")
    w, h = 2, 2
    maze = [[Cell(r, c) for c in range(w)] for r in range(h)]
    maze[0][0].breakWall("E"); maze[0][1].breakWall("W")
    draw_maze(maze, w, h)
    print("--- maze_generation 4x4 + draw_maze ---")
    maze2 = maze_generation(4, 4)
    draw_maze(maze2, 4, 4)
    sol = maze_resolution(maze2)
    print("--- solution ---")
    print(sol)
    print("Format ok:", sol.startswith("(0, 0)") and "+-" in sol)


--- draw_maze 2x2 ---
+--+--+
|     |
+--+--+
|  |  |
+--+--+
--- maze_generation 4x4 + draw_maze ---
+--+--+--+--+
|  |        |
+  +  +--+  +
|     |     |
+--+--+  +--+
|        |  |
+  +--+--+  +
|           |
+--+--+--+--+
--- solution ---
(0, 0)+-S-+(1, 0)+-E-+(1, 1)+-N-+(0, 1)+-E-+(0, 2)+-E-+(0, 3)+-S-+(1, 3)+-W-+(1, 2)+-S-+(2, 2)+-W-+(2, 1)+-W-+(2, 0)+-S-+(3, 0)+-E-+(3, 1)+-E-+(3, 2)+-E-+(3, 3)
Format ok: True


In [ ]:
# lt1
class Octopus:
    def __init__(self):
        self._value = 0.0 # private: float value
        self.p = None # Person p (initialize to None or a Person instance)
        self.n = None # Credit_Card_No (Initialize to None or a instance)
    def Topup_value(self, amount):
    def Pay_Transaction(self, amount):
    def displayAmount(self, amount):

emptyList = list()
emptyList = []
myList = [1, 2, 3, 4, 5]
# heterogenous
mixedList = ["DSAI2201", 2201, 22.01, ["Data Structure"]]


In [2]:
myList = list(range(10))
print(myList)
print(myList[-1:0:-2])

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[9, 7, 5, 3, 1]


In [5]:
>>> stuff = list()
>>> stuff.append('book')
>>> stuff.append(99)
>>> print(stuff)
['book', 99]
>>> stuff.append('cookie')
>>> print(stuff)
['book', 99, 'cookie']

['book', 99]
['book', 99, 'cookie']


['book', 99, 'cookie']

In [13]:
t = ['a', 'b', 'c']
x = t.pop(1) # 如果不写就是pop最后一个
print(t)

t = ['a', 'b', 'c']
del t[1]
print(t)

t = ['a', 'b', 'b', 'c']
t.remove('b')
print(t)

t = ['a', 'b', 'c', 'd', 'e',  'f']
del t[1:5]
print(t)

['a', 'c']
['a', 'c']
['a', 'b', 'c']
['a', 'f']


In [6]:
# adjacency list graph undirected
class Graph:
    def __init__(self): # edged dictionary stores adjacency list
        self.edges = {} # key: vertices, value: adjacent vertices list

    def insertVertex(self, v):
        """add vertices"""
        if v not in self.edges:
            self.edges[v] = []

    def insertEdge(self, u, v):
        """add edges(undirected)"""
        # make sure two vertices exist
        self.insertVertex(u)
        self.insertVertex(v)
        # add v in u's adjacency list(if DNE)
        if v not in self.edges[u]:
            self.edges[u].append(v)
        # add u in v's adjacency list(if DNE)
        if u not in self.edges[v]:
            self.edges[v].append(u)

    def getEdges(self, v):
        """obtain vertex v's all adjacent vertices"""
        return self.edges.get(v, [])

    def areAdjacent(self, u, v):
        """check whether u and v are adjacent(whether have edges)"""
        # check whether two vertices are in the Graph
        if u not in self.edges or v not in self.edges:
            return False
        # Optimization: search vertices list with small degrees
        if len(self.edges[u]) <= len(self.edges[v]):
            return v in self.edges[u]
        else:
            return u in self.edges[v]

# functioning examples 
if __name__ == "__main__":
    g = Graph()
    g.insertEdge('a', 'b')
    g.insertEdge('a', 'c')
    g.insertEdge('b', 'c')
    g.insertEdge('b', 'd')

    print(g.getEdges('b'))
    print(g.areAdjacent('a', 'b'))
    print(g.areAdjacent('a', 'd'))

['a', 'c', 'd']
True
False


In [12]:
# BFS Implemention 
from collections import deque

def bfs(graph, start):
    """
    BFS traversal
    graph: graph represented as adjacency list (dictionary)
    start: starting vertex
    """
    queue = deque()      # Create queue
    visited = set()      # Set of visited vertices
    result = []          # List to store traversal result
    
    queue.append(start)  # Enqueue starting vertex
    visited.add(start)   # Mark as visited
    
    while queue:
        vertex = queue.popleft()  # Dequeue from front
        result.append(vertex) # record traversal order

        # Traversal all adjacent vertices of current vertex
        for neighbor in graph[vertex]:
            if neighbor not in visited:
                visited.add(neighbor) # mark as visited
                queue.append(neighbor) # enqueue for later processing

    return result

# Test case
if __name__ == "__main__":
    # define graph as adjacency list
    graph = {
    'A': ['B', 'C'],
    'B': ['A', 'C', 'D', 'E'],
    'C': ['A', 'B', 'E', 'F'],
    'D': ['B'],
    'E': ['B', 'C'],
    'F': ['C', 'G'],
    'G': ['F']
    }

    # BFS traversal starting from vertex 'A'
    print(bfs(graph, 'A'))

['A', 'B', 'C', 'D', 'E', 'F', 'G']


In [6]:
# BFS最短路径
from collections import deque

def bfs_with_distance(graph, start):
    """
    BFS 并记录距离和前驱
    returns:(distance_dict, predecessor_dict)
    """
    queue = deque()
    distance = {start: 0} # 记录每个顶点到起点的距离
    predecessor = {start: None} # 记录每个顶点的前驱顶点
    queue.append(start)

    while queue:
        vertex = queue.popleft()
        for neighbor in graph[vertex]:
            if neighbor not in distance: # 相当于not visited
                distance[neighbor] = distance[vertex] + 1
                predecessor[neighbor] = vertex
                queue.append(neighbor)

    return distance, predecessor

def get_path(predecessor, satrt, end):
    """"
    根据前驱字典重建从start到end的最短路径
    """
    path = []
    current = end 
    while current is not None:
        path.append(current)
        current = predecessor[current]
    return path[::-1] # 反转得到从起点到终点的路径

# 测试用例
if __name__ == "__main__":
    #定义图的邻接表
    graph = {
        'A': ['B', 'C'],
        'B': ['A', 'C', 'D', 'E'],
        'C': ['A', 'B', 'E', 'F'],
        'D': ['B'],
        'E': ['B', 'C'],
        'F': ['C', 'G'],
        'G': ['F']
    }
    # 执行带距离的BFS
    distance, predecessor = bfs_with_distance(graph, 'A')

    print("各顶点到A的距离:")
    for vertex in sorted(distance.keys()):
        print(f"{vertex}: {distance[vertex]}")
    print("\n从A到G的最短距离：")
    path = get_path(predecessor, 'A', 'G')
    print(path)

    # 验证路径
    print("\n验证路径长度：")
    print(f"路径: {' -> '.join(path)}")  
    print(f"总步数: {len(path)-1}")  

各顶点到A的距离:
A: 0
B: 1
C: 1
D: 2
E: 2
F: 2
G: 3

从A到G的最短距离：
['A', 'C', 'F', 'G']

验证路径长度：
路径: A -> C -> F -> G
总步数: 3


In [8]:
from collections import deque

def bfs_single(graph, start, visited):
    """
    从指定起点执行单次BFS
    graph:邻接表表示的图
    start:起始顶点
    visited:已访问顶点集合（会被修改）
    Returns:当前连通分量的所有顶点列表
    """
    queue = deque()
    queue.append(start)
    visited.add(start)
    component = []

    while queue:
        vertex = queue.popleft()
        component.append(vertex)

        for neighbor in graph[vertex]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return component

def bfs_all_components(graph):
    """
    处理不连通图：多次BFS遍历所有连通分量
    Returns: 所有连通分量的列表
    """
    visited = set()
    components = []

    for vertex in graph:
        if vertex not in visited:
            # 从未访问的顶点启动BFS
            component = bfs_single(graph, vertex, visited)
            components.append(component)

    return components

# 测试用例
if __name__ == "__main__":
    # 不连通图的示例
    graph = {
        'A': ['B', 'C'],
        'B': ['A', 'C'],
        'C': ['A', 'B'],
        'D': ['E'],
        'E': ['D'],
        'F': ['G', 'H'],
        'G': ['F', 'H'],
        'H': ['F', 'G'],
        'I': []  # 孤立顶点
    }
    
    components = bfs_all_components(graph)
    
    print(f"图有 {len(components)} 个连通分量:")
    for i, comp in enumerate(components, 1):
        print(f"分量 {i}: {sorted(comp)}")

图有 4 个连通分量:
分量 1: ['A', 'B', 'C']
分量 2: ['D', 'E']
分量 3: ['F', 'G', 'H']
分量 4: ['I']


In [14]:
def dfs(graph, start):
    """
    DFS遍历（迭代版，使用栈）
    graph:邻接表表示的图（字典）
    start:起始顶点
    """
    stack = [start]  # 使用stack代替队列
    visited = {start}  # 已访问顶点集合
    result = []  # 遍历结果列表

    while stack:
        vertex = stack.pop()  # 从stack顶取出（LIFO）
        result.append(vertex)

        # 检查所有邻接顶点
        # 使用reversed保证顺序与递归版接近
        for neighbor in reversed(graph[vertex]):
            if neighbor not in visited:
                visited.add(neighbor)
                stack.append(neighbor)

    return result  


def dfs_recursive(graph, vertex, visited=None):
    """
    DFS遍历（递归版）
    """
    if visited is None:
        visited = set()  
        
    visited.add(vertex)
    result = [vertex]

    for neighbor in graph[vertex]:
        if neighbor not in visited:
            result.extend(dfs_recursive(graph, neighbor, visited))

    return result  

# 测试用例
if __name__ == "__main__":
    # 定义图的邻接表
    graph = {
        'A': ['B', 'C'],
        'B': ['A', 'C', 'D', 'E'],
        'C': ['A', 'B', 'E', 'F'],
        'D': ['B'],
        'E': ['B', 'C'],
        'F': ['C', 'G'],
        'G': ['F']
    }
    
    # 测试迭代版 DFS
    iterative_result = dfs(graph, 'A')
    print(f"迭代版 DFS 结果: {iterative_result}")
    
    # 测试递归版 DFS
    recursive_result = dfs_recursive(graph, 'A')
    print(f"递归版 DFS 结果: {recursive_result}")
    
    # 对比两种实现的结果差异
    print(f"\n两种实现访问顺序不同但都正确:")
    print(f"迭代版顺序: {' -> '.join(iterative_result)}")
    print(f"递归版顺序: {' -> '.join(recursive_result)}")

迭代版 DFS 结果: ['A', 'B', 'D', 'E', 'C', 'F', 'G']
递归版 DFS 结果: ['A', 'B', 'C', 'E', 'F', 'G', 'D']

两种实现访问顺序不同但都正确:
迭代版顺序: A -> B -> D -> E -> C -> F -> G
递归版顺序: A -> B -> C -> E -> F -> G -> D


In [16]:
from collections import deque

def countIslandsDFS(grid):
    """
    计算岛屿数量（DFS 解法）
    grid: 二维网格，'L'表示陆地，'W'表示水
    """
    if not grid or not grid[0]:
        return 0
    
    n = len(grid)      # 行数
    m = len(grid[0])   # 列数
    islands = 0
    visited = [[False] * m for _ in range(n)]
    
    def dfs(r, c):
        """
        DFS 标记整个岛屿
        """
        visited[r][c] = True
        
        # 四个方向：上、右、左、下
        directions = [(-1, 0), (0, 1), (0, -1), (1, 0)]
        
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            # 检查新位置是否在范围内、是陆地且未访问
            if 0 <= nr < n and 0 <= nc < m and \
               grid[nr][nc] == 'L' and not visited[nr][nc]:
                dfs(nr, nc)
    
    # 遍历网格
    for i in range(n):
        for j in range(m):
            # 找到未访问的陆地，启动 DFS
            if grid[i][j] == 'L' and not visited[i][j]:
                dfs(i, j)
                islands += 1
    
    return islands


def countIslandsBFS(grid):
    """
    计算岛屿数量（BFS 解法）
    grid: 二维网格，'L'表示陆地，'W'表示水
    """
    if not grid or not grid[0]:
        return 0
    
    n = len(grid)      # 行数
    m = len(grid[0])   # 列数
    islands = 0
    visited = [[False] * m for _ in range(n)]
    
    def bfs(start_r, start_c):
        """
        BFS 标记整个岛屿
        """
        queue = deque([(start_r, start_c)])
        visited[start_r][start_c] = True
        
        directions = [(-1, 0), (0, 1), (0, -1), (1, 0)]
        
        while queue:
            r, c = queue.popleft()
            
            for dr, dc in directions:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < m and \
                   grid[nr][nc] == 'L' and not visited[nr][nc]:
                    visited[nr][nc] = True
                    queue.append((nr, nc))
    
    # 遍历网格
    for i in range(n):
        for j in range(m):
            if grid[i][j] == 'L' and not visited[i][j]:
                bfs(i, j)
                islands += 1
    
    return islands


# 测试用例
if __name__ == "__main__":
    # 定义网格
    grid = [
        ['L', 'L', 'W', 'W', 'W'],
        ['W', 'L', 'W', 'W', 'L'],
        ['L', 'W', 'W', 'L', 'L'],
        ['W', 'W', 'W', 'W', 'W'],
        ['L', 'W', 'L', 'L', 'W']
    ]
    
    # 可视化网格
    print("网格地图:")
    for row in grid:
        print(' '.join(row))
    print()
    
    # 测试DFS解法
    dfs_result = countIslandsDFS(grid)
    print(f"DFS解法 - 岛屿数量: {dfs_result}")
    
    # 测试BFS解法
    bfs_result = countIslandsBFS(grid)
    print(f"BFS解法 - 岛屿数量: {bfs_result}")

网格地图:
L L W W W
W L W W L
L W W L L
W W W W W
L W L L W

DFS解法 - 岛屿数量: 5
BFS解法 - 岛屿数量: 5


In [17]:
def degree(graph, vertex):
    """
    计算顶点的度数（无向图）
    graph: 邻接表表示的图
    vertex: 目标顶点
    """
    return len(graph[vertex])


def isConnected(graph, u, v):
    """
    判断两个顶点是否连通（是否有路径相连）
    graph: 邻接表表示的图
    u: 起点顶点
    v: 终点顶点
    """
    if u not in graph or v not in graph:
        return False
    
    visited = set()
    
    def dfs(current):
        """深度优先搜索查找路径"""
        if current == v:  # 找到目标顶点
            return True
        
        visited.add(current)
        
        for neighbor in graph[current]:
            if neighbor not in visited:
                if dfs(neighbor):  # 递归搜索
                    return True
        return False
    
    return dfs(u)


# 测试用例
if __name__ == "__main__":
    # 测试 degree 函数
    print("=== 测试 degree 函数 ===")
    graph1 = {'A': ['B', 'C'], 'B': ['A', 'C'], 'C': ['A', 'B']}
    print(f"图结构: {graph1}")
    print(f"顶点A的度数: {degree(graph1, 'A')}")  # 输出: 2
    print(f"顶点B的度数: {degree(graph1, 'B')}")  # 输出: 2
    print(f"顶点C的度数: {degree(graph1, 'C')}")  # 输出: 2
    
    # 测试 isConnected 函数
    print("\n=== 测试 isConnected 函数 ===")
    graph2 = {
        'A': ['B'],
        'B': ['A', 'C'],
        'C': ['B'],
        'D': []  # 孤立顶点
    }
    print(f"图结构: {graph2}")
    
    # 连通的情况
    print(f"A 和 C 是否连通: {isConnected(graph2, 'A', 'C')}")  # 输出: True
    
    # 不连通的情况
    print(f"A 和 D 是否连通: {isConnected(graph2, 'A', 'D')}")  # 输出: False
    
    # 自身连通的情况
    print(f"A 和 A 是否连通: {isConnected(graph2, 'A', 'A')}")  # 输出: True
    
    # 顶点不存在的情况
    print(f"A 和 E 是否连通: {isConnected(graph2, 'A', 'E')}")  # 输出: False

=== 测试 degree 函数 ===
图结构: {'A': ['B', 'C'], 'B': ['A', 'C'], 'C': ['A', 'B']}
顶点A的度数: 2
顶点B的度数: 2
顶点C的度数: 2

=== 测试 isConnected 函数 ===
图结构: {'A': ['B'], 'B': ['A', 'C'], 'C': ['B'], 'D': []}
A 和 C 是否连通: True
A 和 D 是否连通: False
A 和 A 是否连通: True
A 和 E 是否连通: False


In [18]:
def greedy_schedule(jobs):
    # jobs = [(start, finish), ...]
    jobs.sort(key=lambda x: x[1])
    results = []
    last_finish = 0

    for start, finish in jobs:
        if start >= last_finish:
            results.append((start, finish))
            last_finish = finish

    return results
                    

In [19]:
def activity_selection(s, f):
    """选择最大兼容活动集合（贪心）"""
    n = len(s)
    # 按结束时间排序
    activities = sorted(zip(s, f), key=lambda x: x[1])
    selected = [activities[0]]
    
    for i in range(1, n):
        # 当前活动开始时间 >= 上一个被选活动的结束时间
        if activities[i][0] >= selected[-1][1]:
            selected.append(activities[i])
    
    return selected

In [20]:
def fibonacci_bottom_up(n):
    F = [0, 1]
    for i in range(2, n+1):
        F.append(F[i-1] + F[i-2])
    return F[n]

In [21]:
def fibonacci_naive(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci_naive(n-1) + fibonacci_naive(n-2)

In [22]:
def fibonacci_memo(n, memo=None):
    if memo is None:
        memo = {}

    if n in memo:
        return memo[n]

    if n == 0:
        return 0
    if n == 1:
        return 1

    memo[n] = fibonacci_memo(n-1, memo) + fibonacci_memo(n-2, memo)
    return memo[n]
                    

In [23]:
def fibonacci_optimized(n):
    if n <= 1:
        return n
    prev, curr = 0, 1
    for _ in range(2, n+1):
        prev, curr = curr, prev + curr
    return curr

In [24]:
def knapsack_01(weights, values, capacity):
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            exclude = dp[i-1][w]
            include = 0
            if weights[i-1] <= w:
                include = values[i-1] + dp[i-1][w - weights[i-1]]
            dp[i][w] = max(include, exclude)

    return dp[n][capacity]
                    